# Trying to find the correct bandwidth for estimating curvature when noise and reach are unknown

## Overview

For data concentrated on a manifold with reach $\tau$ with noise level $\sigma$ it helps to know these before to measure bandwidth. In 02-lfs-estimation.ipynb we see that picking the bandwidth is crucial



In [6]:
import numpy as np
from scipy.spatial import KDTree
import matplotlib.pyplot as plt
from ipywidgets import (FloatSlider, FloatLogSlider, IntSlider, Checkbox,
                        HBox, VBox, Layout, interactive_output)
from IPython.display import display

# ── Data generation (unchanged) ───────────────────────────────────────────────

def sample_sphere(n, d, N, sigma, rng):
    n = int(n)
    X_clean_low = rng.standard_normal((n, d + 1))
    X_clean_low /= np.linalg.norm(X_clean_low, axis=1, keepdims=True)
    X_emb = np.zeros((n, N))
    X_emb[:, :d + 1] = X_clean_low
    noise = rng.standard_normal((n, N)) * sigma
    return X_emb + noise, X_emb

# ── Local polynomial fit (unchanged) ──────────────────────────────────────────

def local_poly_fit(x_i, neighbors, d):
    N_amb = x_i.shape[0]
    delta = neighbors - x_i
    n_nbr = delta.shape[0]
    n_mono = d * (d + 1) // 2
    if n_nbr < (d + n_mono + 2):
        return None, None, n_nbr

    _, _, Vt = np.linalg.svd(delta, full_matrices=False)
    T = Vt[:d].T
    v = delta @ T
    Phi = np.zeros((n_nbr, n_mono))
    col = 0
    for i in range(d):
        for j in range(i, d):
            Phi[:, col] = v[:, i] * v[:, j]
            col += 1
    Q, _ = np.linalg.qr(T, mode='complete')
    T_perp = Q[:, d:]
    z_normal = delta @ T_perp
    A, _, _, _ = np.linalg.lstsq(Phi, z_normal, rcond=None)
    N_normal = T_perp.shape[1]
    op_norms = []
    for k in range(N_normal):
        II_k = np.zeros((d, d))
        col = 0
        for i in range(d):
            for j in range(i, d):
                val = A[col, k]
                II_k[i, j] = val
                II_k[j, i] = val
                col += 1
        op_norms.append(np.linalg.svd(II_k, compute_uv=False)[0])
    kappa_max = max(op_norms) if op_norms else 0.0
    tau_hat = 1.0 / kappa_max if kappa_max > 1e-10 else 10.0
    residuals = z_normal - (Phi @ A)
    sigma_hat = np.sqrt(np.mean(residuals**2))
    return tau_hat, sigma_hat, n_nbr

# ── Iterative bandwidth selection (unchanged) ─────────────────────────────────

def iterative_bandwidth(data, d, N, C=5.0, tol=0.02, max_iter=15):
    n = data.shape[0]
    tree = KDTree(data)
    k0 = max(d + 5, int(np.log(n) * 2))
    dist, _ = tree.query(data, k=k0)
    h = dist[:, -1].copy()
    h_star, tau_out, sig_out = np.full(n, np.nan), np.full(n, np.nan), np.full(n, np.nan)
    iter_counts = np.zeros(n, dtype=int)
    constant_factor = C * np.sqrt(N)
    for i in range(n):
        h_i = h[i]
        tau_i, sig_i = 2.0, 0.0 # Initial defaults
        for t in range(max_iter):
            indices = tree.query_ball_point(data[i], h_i)
            if len(indices) < (d + (d*(d+1)//2) + 2):
                h_i *= 1.2
                continue
            tau_i, sig_i, _ = local_poly_fit(data[i], data[indices], d)
            if tau_i is None:
                h_i *= 1.2
                continue
            h_new = np.sqrt(constant_factor * sig_i * tau_i)
            if h_new < 1e-8: break
            if abs(h_new - h_i) / h_i < tol:
                h_i = h_new
                iter_counts[i] = t + 1
                break
            h_i = h_new
            iter_counts[i] = t + 1
        h_star[i], tau_out[i], sig_out[i] = h_i, tau_i, sig_i
    return h_star, tau_out, sig_out, iter_counts

# ── Visualization (UPDATED FOR LOG SCALE REACH) ───────────────────────────────

def run_simulation(d, N, sigma, n, show_convergence):
    plt.close('all')
    rng = np.random.default_rng(42)
    n = int(n)
    if d >= N:
        print(f"Error: d ({d}) must be < N ({N})")
        return

    data, _ = sample_sphere(n, d, N, sigma, rng)
    h_star, tau_hat, sig_hat, n_iters = iterative_bandwidth(data, d, N)
    h_theory = np.sqrt(5.0 * np.sqrt(N) * sigma)

    ncols = 4 if show_convergence else 3
    fig, axes = plt.subplots(1, ncols, figsize=(4 * ncols, 3.8))
    fig.suptitle(fr"Manifold Geometry: $d={d}, N={N}, n={n}, \sigma={sigma:.3f}$", fontsize=14)

    # Plot 1: Bandwidth
    axes[0].hist(h_star[np.isfinite(h_star)], bins=20, color='skyblue', ec='white')
    axes[0].axvline(h_theory, color='red', ls='--', label=r'Theory $h^*$')
    axes[0].set_title(r'Bandwidth $h^*$')
    axes[0].legend()

    # Plot 2: Reach (Log Scale)
    valid_tau = tau_hat[np.isfinite(tau_hat)]
    if len(valid_tau) > 0:
        t_min, t_max = np.min(valid_tau), np.max(valid_tau)
        # Bins must be log-spaced for a log-scale histogram to look correct
        bins = np.logspace(np.log10(max(t_min, 1e-2)), np.log10(max(t_max, 1.1)), 25)
        axes[1].hist(valid_tau, bins=bins, color='lightgreen', ec='white')
    axes[1].axvline(1.0, color='red', ls='--', label=r'True $\tau=1$')
    axes[1].set_xscale('log')
    axes[1].set_title(r'Reach $\hat{\tau}$ (Log Scale)')
    axes[1].legend()

    # Plot 3: Noise
    axes[2].hist(sig_hat[np.isfinite(sig_hat)], bins=20, color='salmon', ec='white')
    axes[2].axvline(sigma, color='black', ls='--', label=r'True $\sigma$')
    axes[2].set_title(r'Noise $\hat{\sigma}$')
    axes[2].legend()

    if show_convergence:
        s_vals = np.linspace(0.01, 0.12, 10)
        m_iters = [np.mean(iterative_bandwidth(sample_sphere(min(n, 150), d, N, s, rng)[0], d, N)[3]) for s in s_vals]
        axes[3].plot(s_vals, m_iters, 'o-', color='purple', lw=2)
        print(m_iters)
        axes[3].axvline(sigma, color='black', ls=':', alpha=0.6, label=r'Current $\sigma$')
        axes[3].set_title('Avg Iterations')
        axes[3].set_xlabel(r'$\sigma$')
        axes[3].legend()

    plt.tight_layout()
    plt.show()

# ── UI Construction (95% width) ───────────────────────────────────────────────

s_layout = Layout(width='95%')
s_style = {'description_width': 'initial'}

w_d = IntSlider(min=1, max=8, value=2, description=r'Intrinsic dimension ($d$):', style=s_style, layout=s_layout)
w_N = IntSlider(min=3, max=20, value=6, description=r'Ambient dimension ($N$):', style=s_style, layout=s_layout)
w_sigma = FloatSlider(min=0.01, max=0.15, step=0.01, value=0.03, description=r'Noise level ($\sigma$):', style=s_style, layout=s_layout)
w_n = FloatLogSlider(min=2, max=3.3, step=0.1, value=400, description=r'Points ($n$):', style=s_style, layout=s_layout)
w_conv = Checkbox(value=False, description='Show convergence plot (adds latency)', indent=False)

ui = VBox([w_d, w_N, w_sigma, w_n, w_conv])
out = interactive_output(run_simulation, {'d': w_d, 'N': w_N, 'sigma': w_sigma, 'n': w_n, 'show_convergence': w_conv})

display(ui, out)

Output()